# 033 · Exponentially Weighted Moving Average (EWMA)

A detour, and the most useful one in the module. EWMA is **not an optimizer** —
it is a way of extracting the trend from noisy data using **one stored number**.
Momentum, RMSprop and Adam are all built on it.

$$V_t = \beta V_{t-1} + (1-\beta)\theta_t, \qquad V_0 = 0$$

| Part | What we reproduce |
|---|---|
| A | the formula, and the weighted history hidden inside it |
| B | effective window ≈ **1/(1−β)**: 0.5 → ~2 points, 0.9 → ~10, 0.98 → ~50 |
| C | **higher β is smoother AND laggier** — measured |
| D | the cold start, and the **1/(1−βᵗ)** correction Adam uses |

Needs `numpy`.

In [ ]:
import numpy as np

rng = np.random.default_rng(4)
T = 100
series = 22 + 8 * np.sin(2 * np.pi * np.arange(T) / 60) + rng.normal(0, 2.5, T)

print("A noisy daily temperature series:", T, "points")
print("first 8 values:", np.round(series[:8], 1))

## Part A — One line, one stored number

In [ ]:
def ewma(x, beta, correct=False):
    v, out = 0.0, []
    for t, theta in enumerate(x, 1):
        v = beta * v + (1 - beta) * theta
        out.append(v / (1 - beta ** t) if correct else v)
    return np.array(out)


smooth = ewma(series, 0.9)
print(f"{'t':>4}{'raw':>9}{'ewma(0.9)':>12}")
for t in (0, 1, 2, 10, 50, 99):
    print(f"{t+1:>4}{series[t]:>9.2f}{smooth[t]:>12.2f}")

print("\nThe state is a single float. No window buffer, no history array.")

In [ ]:
# Yet it IS a weighted history - expand the recursion and the weights appear.
beta = 0.9
print("weight this EWMA places on each of the last 8 observations:\n")
for age in range(8):
    print(f"  {age} steps back: {(1 - beta) * beta ** age:.5f}")

total = sum((1 - beta) * beta ** age for age in range(200))
print(f"\nweights sum to {total:.4f} (they sum to 1 in the limit)")
print("Each step back multiplies by another beta - the decay is exponential.")
print("You get a weighted history of EVERYTHING while storing one number.")

## Part B — β sets the effective window

The rule of thumb is `1/(1−β)`. Check it: find how many of the most recent points
carry most of the weight.

In [ ]:
print(f"{'beta':>7}{'1/(1-beta)':>13}{'points for 86% of weight':>27}")
for b in (0.5, 0.9, 0.98):
    w = [(1 - b) * b ** age for age in range(500)]
    cum = np.cumsum(w)
    k = int(np.searchsorted(cum, 0.86)) + 1        # 1 - 1/e is about 0.632... use 86%
    print(f"{b:>7}{1/(1-b):>13.0f}{k:>27}")

print("\nThe rule of thumb tracks the real weight mass closely enough to trust.")

## Part C — Smoother *and* laggier

This is the trade, and it is the reason β is a knob rather than a constant.
Measure the lag directly: shift the smoothed curve until it best lines up with
the underlying signal.

In [ ]:
clean = 22 + 8 * np.sin(2 * np.pi * np.arange(T) / 60)      # the signal without noise

def lag_of(smoothed, ref, max_lag=40):
    """How far to shift `smoothed` back to best match `ref`."""
    best, best_lag = -np.inf, 0
    for k in range(max_lag):
        a = smoothed[k:]
        b = ref[:len(a)]
        c = np.corrcoef(a, b)[0, 1]
        if c > best:
            best, best_lag = c, k
    return best_lag


print(f"{'beta':>7}{'noise left (std)':>19}{'lag (steps)':>14}")
for b in (0.0, 0.5, 0.9, 0.98):
    s = ewma(series, b)
    residual_noise = float(np.std(s - clean))
    print(f"{b:>7}{residual_noise:>19.3f}{lag_of(s, clean):>14}")

print("\nMore smoothing, more delay. There is no setting that gives both.")

In [ ]:
s98 = ewma(series, 0.98)
peak_true = int(np.argmax(clean[:70]))
peak_98 = int(np.argmax(s98[:70]))
print(f"the underlying signal peaks at t = {peak_true}")
print(f"the beta=0.98 curve peaks at    t = {peak_98}")
print(f"-> {peak_98 - peak_true} steps late")
print("\nAt beta = 0.98 the curve is still rising after the data has turned.")
assert peak_98 > peak_true

## Part D — The cold start, and the fix Adam uses

`V₀ = 0` is a lie: it asserts the series started at zero. So the first few values
are dragged down. `V₁` is only `(1−β)θ₁` — at β = 0.98, that is **2%** of the
first observation.

In [ ]:
b = 0.98
raw = ewma(series, b)
fixed = ewma(series, b, correct=True)

print(f"true first value: {series[0]:.2f}\n")
print(f"{'t':>4}{'uncorrected':>14}{'corrected':>12}{'1/(1-b^t)':>12}")
for t in (0, 1, 2, 5, 20, 99):
    print(f"{t+1:>4}{raw[t]:>14.3f}{fixed[t]:>12.3f}{1/(1-b**(t+1)):>12.2f}")

print(f"\nV1 is only (1-beta) * theta1 = {(1-b) * series[0]:.3f}, or {1-b:.0%} of it.")

In [ ]:
print("The correction divides by 1 - beta^t, which starts large and tends to 1:")
for t in (1, 2, 10, 100, 1000):
    print(f"  t = {t:>5}: factor = {1/(1-b**t):>8.2f}")

print("\nThis is EXACTLY what Adam does to its two moment estimates.")
print("You will meet it again in lesson 038 - it is not optional there.")

assert abs(fixed[0] - series[0]) < 1e-9      # the correction is exact at t=1

## What to take away

- **EWMA extracts the trend from noisy time-series data.** It is a tool, not an
  optimizer.
- **`V_t = βV_{t−1} + (1−β)θ_t`**, with `V₀ = 0`. One line, one stored number.
- **Weights decay exponentially with age** — each step back multiplies by
  another β.
- **You get a weighted history of everything while storing only the previous
  value.**
- **Effective window ≈ 1/(1−β)**: 0.5 → ~2 points, 0.9 → ~10, 0.98 → ~50.
- **Higher β is smoother AND laggier.** At 0.98 the curve is still rising after
  the data has turned.
- **Cold start:** `V₀ = 0` biases the first values low — `V₁` is only
  `(1−β)θ₁`.
- **Dividing by `1 − βᵗ` corrects that bias** — exactly what Adam does.
- **Momentum EWMAs the gradient; RMSprop EWMAs its square; Adam does both.**

## Exercises

1. Derive the closed form `V_t = (1−β) Σ βᵏ θ_{t−k}` from the recursion, and
   check it numerically against the loop for β = 0.9, t = 20.
2. The "effective window" rule `1/(1−β)` is a rule of thumb. Work out the β for
   which exactly half the weight sits in the last 5 observations.
3. Part C measured lag by cross-correlation. Measure it a second way — the delay
   of the peak — and see whether the two agree.
4. Bias correction changes the early values a lot. Plot corrected against
   uncorrected for β = 0.9 and β = 0.999. For which is it more urgent, and why
   does that matter for Adam's `β₂ = 0.999`?
5. Apply EWMA to a series with a **step change** rather than a smooth trend. How
   many steps does each β take to catch up? What does that suggest about using
   high β on a loss that suddenly drops?